# Data pre-processing
### Step 1: Import necessary libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
train = pd.read_csv ("/kaggle/input/datasets/adhamad0/spaceship-train/train.csv")
train

### Step 2: Drop irrelevant columns

'PassengerId' and 'Name' are only identifiers and have no predictive value.

In [ ]:
train = train.drop (columns = ['PassengerId', 'Name'])

train

### Step 3: Handle missing values

- numerical data use median imputation
- categorical/boolean data use mode imputation

In [ ]:
# Split the columns into numerical or categorical
num_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
cat_cols = ["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]

# Fill in missing values for numerical data
train [num_cols] = train [num_cols].fillna (train [num_cols].median ())

# Fill in missing values for categorical data
for col in cat_cols:
    train [col] = train [col].fillna (train [col].mode ()[0])

train

### Step 4: Convert boolean values into numerical values

In [ ]:
train ["CryoSleep"] = train ["CryoSleep"].astype (int)
train ["VIP"] = train ["VIP"].astype (int)
train ["Transported"] = train ["Transported"].astype (int)

train

### Step 5: Handle cabin

In [ ]:
train [["Deck", "Num", "Side"]] = train ["Cabin"].str.split ("/", expand = True)
train = train.drop (columns = "Cabin")

train

### Step 6: Convert categorical data into numerical data
- One-hot encoding

In [ ]:
cat_cols = ["HomePlanet", "Destination", "Deck", "Side"]
train = pd.get_dummies (train, columns = cat_cols, drop_first = True)

train

### Step 7: Separate X and y

In [ ]:
X = train.drop (columns = "Transported")
y = train ["Transported"]

### Step 8: Feature Scaling
- Z-score standardisation

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler ()
X_scaled = scaler.fit_transform (X)

# Model selection
### Logistic Regression (forward stepwise)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_scaled = pd.DataFrame (X_scaled, columns = X.columns)

X_train, X_val, y_train, y_val = train_test_split (X_scaled, y, test_size = 0.2, random_state = 42)

features = list (X.columns)
selected_features = []
remaining_features = features.copy ()
best_score = 0

while len (remaining_features) > 0:
    scores = []

    for feature in remaining_features:
        current_features = selected_features + [feature]

        model = LogisticRegression (max_iter = 1000)
        model.fit (X_train [current_features], y_train)

        y_pred = model.predict (X_val [current_features])
        accuracy = accuracy_score (y_val, y_pred)
        
        scores.append ((accuracy, feature))
    
    scores.sort (reverse = True)
    best_new_score, best_feature = scores [0]

    if best_new_score <= best_score:
        break

    selected_features.append (best_feature)
    remaining_features.remove (best_feature)
    best_score = best_new_score

print ("Final accuracy score: ", best_score)
print ("Selected features: ", selected_features)

# Test data loading